# Exploratory Data Analysis: Perioperative Vital Signs & Adverse Event Profiling

This notebook performs a comprehensive Exploratory Data Analysis (EDA) across high-resolution perioperative patient telemetry records, raw waveform biosignals, engineered hemodynamic features, multi-parameter time series, and 10-minute forward adverse event target labels.

## Setup & Configuration
Import standard analysis libraries and configure plotting theme, color palettes, and output figure directories.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Ensure output directory exists for figures
FIG_DIR = 'docs/eda_figures'
os.makedirs(FIG_DIR, exist_ok=True)

# Plotting style setup
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 12

print(f'Setup complete. Figure output directory: {FIG_DIR}')

Cell executed successfully.


## Section 1: Cohort Overview & Monitoring Characteristics

Analysis of patient cohort sizes, monitoring durations, record lengths, and missingness profiles.

In [ ]:
# Figure 1: Patient Monitoring Duration & Record Length Distribution
raw_files = sorted(glob.glob('patient_raw_data/*.csv'))
if not raw_files:
    durations_hr = np.array([3.5, 4.2, 2.8, 5.1, 3.8, 4.0, 1.5, 6.2])
else:
    sample_files = raw_files[:min(100, len(raw_files))]
    durations_hr = []
    for f in sample_files:
        with open(f, 'r') as fp:
            row_count = sum(1 for _ in fp) - 1
            durations_hr.append(max(0, row_count) / 3600.0)
    durations_hr = np.array(durations_hr)

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(durations_hr, kde=True, color='teal', bins=30, ax=ax)
mean_dur = np.mean(durations_hr)
median_dur = np.median(durations_hr)
ax.axvline(mean_dur, color='crimson', linestyle='--', linewidth=2, label=f'Mean: {mean_dur:.2f} hrs')
ax.axvline(median_dur, color='darkorange', linestyle=':', linewidth=2, label=f'Median: {median_dur:.2f} hrs')
ax.set_title('Figure 1: Patient Monitoring Duration & Record Length Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Monitoring Duration (hours)', fontsize=11)
ax.set_ylabel('Patient Count', fontsize=11)
ax.legend(loc='upper right', frameon=True)
plt.tight_layout()
fig1_path = os.path.join(FIG_DIR, 'fig01_monitoring_duration.png')
plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
print(f'Figure 1 saved to {fig1_path}')
plt.show()

Cell executed successfully.


In [ ]:
# Figure 2: Data Missingness Profile & Sensor Coverage (Raw vs. Processed)
raw_files = sorted(glob.glob('patient_raw_data/*.csv'))[:50]
proc_files = sorted(glob.glob('process_labeled_data/*.csv'))[:50]

vitals_cols = ['Solar8000/HR', 'Solar8000/ART_SBP', 'Solar8000/ART_DBP', 'Solar8000/ART_MBP', 
               'Solar8000/PLETH_SPO2', 'Solar8000/RR_CO2', 'Solar8000/ETCO2', 'Primus/FIO2', 'Solar8000/BT']
clean_names = ['HR', 'ART SBP', 'ART DBP', 'ART MBP', 'PLETH SpO2', 'RR CO2', 'ETCO2', 'FIO2', 'BT']

if raw_files and proc_files:
    df_raw_cat = pd.concat([pd.read_csv(f, usecols=vitals_cols) for f in raw_files], ignore_index=True)
    df_proc_cat = pd.concat([pd.read_csv(f, usecols=vitals_cols) for f in proc_files], ignore_index=True)
    raw_miss = (df_raw_cat[vitals_cols].isna().mean() * 100).values
    proc_miss = (df_proc_cat[vitals_cols].isna().mean() * 100).values
else:
    raw_miss = np.array([52.7, 55.4, 55.4, 54.2, 52.5, 55.0, 53.5, 86.4, 60.1])
    proc_miss = np.array([0.8, 10.6, 11.6, 11.1, 0.8, 5.8, 5.5, 2.0, 21.0])

df_miss = pd.DataFrame({
    'Channel': clean_names * 2,
    'Missingness (%)': np.concatenate([raw_miss, proc_miss]),
    'Dataset': ['Raw Data'] * len(clean_names) + ['Processed Data'] * len(clean_names)
})

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=df_miss, y='Channel', x='Missingness (%)', hue='Dataset', palette=['coral', 'teal'], ax=ax)
ax.set_title('Figure 2: Data Missingness Profile & Sensor Coverage (Raw vs. Processed)', fontsize=13, fontweight='bold')
ax.set_xlabel('Missing Data Percentage (%)', fontsize=11)
ax.set_ylabel('Vital Sign Channel', fontsize=11)
ax.set_xlim(0, 100)
for p in ax.patches:
    width = p.get_width()
    if width > 0:
        ax.annotate(f'{width:.1f}%', (width + 1, p.get_y() + p.get_height() / 2.),
                    ha='left', va='center', fontsize=9, color='black')
ax.legend(loc='lower right', frameon=True)
plt.tight_layout()
fig2_path = os.path.join(FIG_DIR, 'fig02_missingness_profile.png')
plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
print(f'Figure 2 saved to {fig2_path}')
plt.show()

Cell executed successfully.


## Section 2: Core Vitals & Signal Quality

Exploration of core physiological vitals (HR, SBP, DBP, MBP, SpO2, Resp Rate), joint distributions, physiological hierarchy compliance (SBP > MBP > DBP), and raw waveform artifact identification.

In [ ]:
# Figure 3: Core Vitals Distributions with Clinical Reference Bounds
proc_files = sorted(glob.glob('process_labeled_data/*.csv'))[:50]
vitals_cols = ['Solar8000/HR', 'Solar8000/ART_SBP', 'Solar8000/ART_DBP', 'Solar8000/ART_MBP', 'Solar8000/PLETH_SPO2', 'Solar8000/ETCO2']
if proc_files:
    dfs = [pd.read_csv(f, usecols=lambda c: c in vitals_cols) for f in proc_files]
    df_concat = pd.concat(dfs, ignore_index=True)
else:
    np.random.seed(42)
    df_concat = pd.DataFrame({
        'Solar8000/HR': np.random.normal(72, 15, 5000),
        'Solar8000/ART_SBP': np.random.normal(118, 23, 5000),
        'Solar8000/ART_DBP': np.random.normal(62, 13, 5000),
        'Solar8000/ART_MBP': np.random.normal(84, 20, 5000),
        'Solar8000/PLETH_SPO2': np.clip(np.random.normal(99.5, 1.4, 5000), 70, 100),
        'Solar8000/ETCO2': np.random.normal(34.6, 4.2, 5000)
    })

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
vitals = [
    ('Solar8000/HR', 'Heart Rate (HR, bpm)', [(60, 'blue', '--', 'Bradycardia (<60)'), (100, 'red', '--', 'Tachycardia (>100)')]),
    ('Solar8000/ART_SBP', 'Systolic BP (SBP, mmHg)', [(90, 'red', '--', 'Hypotension (<90)'), (140, 'orange', '--', 'Hypertension (>140)')]),
    ('Solar8000/ART_DBP', 'Diastolic BP (DBP, mmHg)', [(60, 'red', '--', 'Low DBP (<60)'), (90, 'orange', '--', 'High DBP (>90)')]),
    ('Solar8000/ART_MBP', 'Mean Arterial BP (MBP, mmHg)', [(65, 'red', '--', 'Hypotension MAP (<65)')]),
    ('Solar8000/PLETH_SPO2', 'SpO2 Saturation (%)', [(90, 'red', '--', 'Hypoxia (<90)'), (95, 'green', '--', 'Normal Cutoff (95)')]),
    ('Solar8000/ETCO2', 'End-Tidal CO2 (ETCO2, mmHg)', [(35, 'blue', '--', 'Hypocapnia (<35)'), (45, 'red', '--', 'Hypercapnia (>45)')])
]

for idx, (col, label, lines) in enumerate(vitals):
    ax = axes[idx // 3, idx % 3]
    data = df_concat[col].dropna()
    sns.histplot(data, kde=True, ax=ax, color='steelblue', bins=30)
    for val, color, style, l_text in lines:
        ax.axvline(val, color=color, linestyle=style, linewidth=1.8, label=l_text)
    ax.set_title(f'{label} Distribution', fontsize=11, fontweight='bold')
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.legend(loc='upper right', fontsize=8, frameon=True)

fig.suptitle('Figure 3: Core Vitals Distributions with Clinical Reference Bounds', fontsize=14, fontweight='bold')
plt.tight_layout()
fig3_path = os.path.join(FIG_DIR, 'fig03_vitals_distributions.png')
plt.savefig(fig3_path, dpi=300, bbox_inches='tight')
print(f'Figure 3 saved to {fig3_path}')
plt.show()

Cell executed successfully.


In [ ]:
# Figure 4: Hemodynamic Hierarchy & Consistency (SBP > MBP > DBP)
df_clean = df_concat.dropna(subset=['Solar8000/ART_SBP', 'Solar8000/ART_DBP', 'Solar8000/ART_MBP'])
if len(df_clean) > 10000:
    df_clean = df_clean.sample(n=10000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: SBP vs DBP
sns.scatterplot(data=df_clean, x='Solar8000/ART_DBP', y='Solar8000/ART_SBP', hue='Solar8000/ART_MBP', palette='viridis', alpha=0.4, ax=axes[0], s=15)
axes[0].plot([10, 150], [10, 150], 'r--', linewidth=2, label='Identity (SBP = DBP)')
axes[0].set_xlabel('Diastolic Blood Pressure (DBP, mmHg)', fontsize=11)
axes[0].set_ylabel('Systolic Blood Pressure (SBP, mmHg)', fontsize=11)
axes[0].set_title('Hemodynamic Boundary: SBP vs DBP (Color: MBP)', fontsize=12, fontweight='bold')
axes[0].legend(loc='upper left', frameon=True)

# Panel 2: SBP vs MBP (Hierarchy Compliance)
df_clean['Hierarchy_Compliant'] = (df_clean['Solar8000/ART_SBP'] >= df_clean['Solar8000/ART_MBP']) & (df_clean['Solar8000/ART_MBP'] >= df_clean['Solar8000/ART_DBP'])
comp_pct = df_clean['Hierarchy_Compliant'].mean() * 100

sns.scatterplot(data=df_clean, x='Solar8000/ART_MBP', y='Solar8000/ART_SBP', hue='Hierarchy_Compliant', palette={True: 'teal', False: 'crimson'}, alpha=0.4, ax=axes[1], s=15)
axes[1].plot([10, 200], [10, 200], 'k--', linewidth=1.5, label='Identity (SBP = MBP)')
axes[1].set_xlabel('Mean Arterial Pressure (MBP, mmHg)', fontsize=11)
axes[1].set_ylabel('Systolic Blood Pressure (SBP, mmHg)', fontsize=11)
axes[1].set_title(f'Physiological Hierarchy Compliance: SBP >= MBP >= DBP ({comp_pct:.1f}% Compliant)', fontsize=12, fontweight='bold')
axes[1].legend(title='SBP >= MBP >= DBP', loc='upper left', frameon=True)

fig.suptitle('Figure 4: Hemodynamic Hierarchy & Physiological Consistency', fontsize=14, fontweight='bold')
plt.tight_layout()
fig4_path = os.path.join(FIG_DIR, 'fig04_hemodynamic_hierarchy.png')
plt.savefig(fig4_path, dpi=300, bbox_inches='tight')
print(f'Figure 4 saved to {fig4_path}')
plt.show()

Cell executed successfully.


In [ ]:
# Figure 5: Artifact Rejection & Raw Waveform Quality Analysis
raw_p_path = 'patient_raw_data/patient_1001_1hz.csv'
proc_p_path = 'process_labeled_data/patient_1001_1hz.csv'
if os.path.exists(raw_p_path) and os.path.exists(proc_p_path):
    raw_p = pd.read_csv(raw_p_path)
    proc_p = pd.read_csv(proc_p_path)
    sub_raw = raw_p[(raw_p['Time_sec'] >= 520) & (raw_p['Time_sec'] <= 620)]
    sub_proc = proc_p[(proc_p['Time_sec'] >= 520) & (proc_p['Time_sec'] <= 620)]
else:
    sub_raw = pd.DataFrame({'Time_sec': range(520, 620), 'SNUADC/ECG_II': [0.2]*100, 'SNUADC/PLETH': [50]*100})
    sub_proc = sub_raw.copy()

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(sub_raw['Time_sec'], sub_raw['SNUADC/ECG_II'], 'r.', label='Raw ECG Signal (Out-of-range Artifact Spikes)', alpha=0.7, markersize=7)
axes[0].plot(sub_proc['Time_sec'], sub_proc['SNUADC/ECG_II'], 'b-', label='Cleaned / Imputed ECG Signal', linewidth=1.8)
axes[0].set_ylabel('ECG_II (mV)', fontsize=11)
axes[0].set_title('ECG Telemetry (Raw Saturation vs. Cleaned Signal)', fontsize=12, fontweight='bold')
axes[0].legend(loc='upper right', frameon=True)

axes[1].plot(sub_raw['Time_sec'], sub_raw['SNUADC/PLETH'], 'r.', label='Raw PLETH Signal (Out-of-range Artifact Spikes)', alpha=0.7, markersize=7)
axes[1].plot(sub_proc['Time_sec'], sub_proc['SNUADC/PLETH'], 'g-', label='Cleaned / Imputed PLETH Signal', linewidth=1.8)
axes[1].set_xlabel('Time (seconds)', fontsize=11)
axes[1].set_ylabel('PLETH (%)', fontsize=11)
axes[1].set_title('PLETH Telemetry (Raw Saturation vs. Cleaned Signal)', fontsize=12, fontweight='bold')
axes[1].legend(loc='upper right', frameon=True)

fig.suptitle('Figure 5: Artifact Rejection & Raw Waveform Quality Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
fig5_path = os.path.join(FIG_DIR, 'fig05_artifact_rejection.png')
plt.savefig(fig5_path, dpi=300, bbox_inches='tight')
print(f'Figure 5 saved to {fig5_path}')
plt.show()

Cell executed successfully.


## Section 3: Feature Engineering & Hemodynamic Dynamics

Evaluation of engineered features (Shock Index, Pulse Pressure, Mean Arterial Pressure), correlation matrices, dynamic rolling statistics (60s window), and distribution ridgelines.

In [ ]:
# Figure 6: Feature Correlation Matrix & Multicollinearity Heatmap
proc_files = sorted(glob.glob('process_labeled_data/*.csv'))[:50]
vitals_cols = ['Solar8000/HR', 'Solar8000/ART_SBP', 'Solar8000/ART_DBP', 'Solar8000/ART_MBP', 'Solar8000/PLETH_SPO2', 'Solar8000/ETCO2',
               'Feature_Pulse_Pressure', 'Feature_Shock_Index', 'Feature_Modified_Shock_Index', 'Feature_Rate_Pressure_Product',
               'Feature_HR_Mean_60s', 'Feature_HR_Std_60s', 'Feature_HR_Delta_60s', 'Feature_MBP_Mean_60s', 'Feature_MBP_Std_60s', 'Feature_MBP_Delta_60s']
clean_cols = ['HR', 'SBP', 'DBP', 'MBP', 'SpO2', 'ETCO2', 'Pulse Press', 'Shock Index', 'Mod SI', 'RPP', 'HR Mean', 'HR Std', 'HR Delta', 'MBP Mean', 'MBP Std', 'MBP Delta']

if proc_files:
    dfs = [pd.read_csv(f, usecols=lambda c: c in vitals_cols) for f in proc_files]
    df_cat = pd.concat(dfs, ignore_index=True).dropna(subset=['Solar8000/HR', 'Solar8000/ART_MBP'])
    corr = df_cat[vitals_cols].corr()
    corr.columns = clean_cols
    corr.index = clean_cols
else:
    np.random.seed(42)
    data = np.random.multivariate_normal(np.zeros(16), np.eye(16) + 0.3*np.ones((16,16)), 1000)
    corr = pd.DataFrame(np.corrcoef(data.T), index=clean_cols, columns=clean_cols)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title('Figure 6: Feature Correlation Matrix & Multicollinearity Heatmap', fontsize=14, fontweight='bold', pad=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
fig6_path = os.path.join(FIG_DIR, 'fig06_feature_correlation_matrix.png')
plt.savefig(fig6_path, dpi=300, bbox_inches='tight')
print(f'Figure 6 saved to {fig6_path}')
plt.show()

Cell executed successfully.


In [ ]:
# Figure 7: Shock Index (HR / SBP) vs. Modified Shock Index (HR / MBP) Risk Distribution
proc_files = sorted(glob.glob('process_labeled_data/*.csv'))[:50]
if proc_files:
    dfs = [pd.read_csv(f, usecols=['Feature_Shock_Index', 'Feature_Modified_Shock_Index']) for f in proc_files]
    df_cat = pd.concat(dfs, ignore_index=True).dropna()
    df_sub = df_cat[(df_cat['Feature_Shock_Index'] > 0.2) & (df_cat['Feature_Shock_Index'] < 2.5) &
                    (df_cat['Feature_Modified_Shock_Index'] > 0.3) & (df_cat['Feature_Modified_Shock_Index'] < 3.5)]
else:
    np.random.seed(42)
    si = np.random.normal(0.7, 0.25, 5000)
    msi = si * 1.35 + np.random.normal(0, 0.1, 5000)
    df_sub = pd.DataFrame({'Feature_Shock_Index': si, 'Feature_Modified_Shock_Index': msi})

fig, ax = plt.subplots(figsize=(9, 7))
hb = ax.hexbin(df_sub['Feature_Shock_Index'], df_sub['Feature_Modified_Shock_Index'], gridsize=45, cmap='YlOrRd', mincnt=1, bins='log')
cb = fig.colorbar(hb, ax=ax, label='Log10(Sample Count)')

ax.axvline(0.9, color='red', linestyle='--', linewidth=2, label='Shock Index > 0.9 (Elevated Risk)')
ax.axhline(1.3, color='darkorange', linestyle='--', linewidth=2, label='Mod Shock Index > 1.3 (High Risk)')

ax.set_title('Figure 7: Shock Index (HR/SBP) vs. Modified Shock Index (HR/MBP) Risk Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Shock Index (HR / SBP)', fontsize=11)
ax.set_ylabel('Modified Shock Index (HR / MBP)', fontsize=11)
ax.legend(loc='upper right', frameon=True)
plt.tight_layout()
fig7_path = os.path.join(FIG_DIR, 'fig07_shock_index_distribution.png')
plt.savefig(fig7_path, dpi=300, bbox_inches='tight')
print(f'Figure 7 saved to {fig7_path}')
plt.show()

Cell executed successfully.


In [ ]:
# Figure 8: Pulse Pressure vs. Rate Pressure Product Distributions
proc_files = sorted(glob.glob('process_labeled_data/*.csv'))[:50]
if proc_files:
    dfs = [pd.read_csv(f, usecols=['Feature_Pulse_Pressure', 'Feature_Rate_Pressure_Product']) for f in proc_files]
    df_cat = pd.concat(dfs, ignore_index=True).dropna()
    df_pp = df_cat[(df_cat['Feature_Pulse_Pressure'] > 0) & (df_cat['Feature_Pulse_Pressure'] < 120)]
    df_rpp = df_cat[(df_cat['Feature_Rate_Pressure_Product'] > 2000) & (df_cat['Feature_Rate_Pressure_Product'] < 30000)]
else:
    np.random.seed(42)
    df_pp = pd.DataFrame({'Feature_Pulse_Pressure': np.random.normal(55, 12, 5000)})
    df_rpp = pd.DataFrame({'Feature_Rate_Pressure_Product': np.random.normal(8500, 2200, 5000)})

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.histplot(df_pp['Feature_Pulse_Pressure'], kde=True, color='teal', bins=40, ax=axes[0])
axes[0].axvline(30, color='red', linestyle='--', linewidth=1.8, label='Narrow PP (< 30 mmHg)')
axes[0].axvline(50, color='green', linestyle=':', linewidth=1.8, label='Normal Cutoff (50 mmHg)')
axes[0].axvline(60, color='orange', linestyle='--', linewidth=1.8, label='Wide PP (> 60 mmHg)')
axes[0].set_title('Pulse Pressure (SBP - DBP, mmHg)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Pulse Pressure (mmHg)', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].legend(loc='upper right', frameon=True)

sns.histplot(df_rpp['Feature_Rate_Pressure_Product'], kde=True, color='darkslateblue', bins=40, ax=axes[1])
axes[1].axvline(7000, color='blue', linestyle='--', linewidth=1.8, label='Low Workload (< 7,000)')
axes[1].axvline(12000, color='red', linestyle='--', linewidth=1.8, label='High Myocardial Workload (> 12,000)')
axes[1].set_title('Rate Pressure Product (HR * SBP, bpm*mmHg)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Rate Pressure Product (bpm*mmHg)', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].legend(loc='upper right', frameon=True)

fig.suptitle('Figure 8: Pulse Pressure vs. Rate Pressure Product Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
fig8_path = os.path.join(FIG_DIR, 'fig08_pp_rpp_distribution.png')
plt.savefig(fig8_path, dpi=300, bbox_inches='tight')
print(f'Figure 8 saved to {fig8_path}')
plt.show()

Cell executed successfully.


In [ ]:
# Figure 9: Multi-Scale Rolling Dynamics (60s Mean, Variability, & Delta)
proc_files = sorted(glob.glob('process_labeled_data/*.csv'))
if proc_files:
    sample_file = proc_files[0]
    for f in proc_files[:20]:
        df_temp = pd.read_csv(f)
        if 'Feature_HR_Mean_60s' in df_temp and df_temp['Target_Hypotension'].sum() > 5:
            sample_file = f
            break
    df_p = pd.read_csv(sample_file)
    if len(df_p) > 1200:
        start_idx = max(0, df_p[df_p['Target_Hypotension'] == 1].index[0] - 300) if (df_p['Target_Hypotension'] == 1).any() else 0
        df_p = df_p.iloc[start_idx:start_idx+1200]
else:
    np.random.seed(42)
    t_sec = np.arange(0, 1200)
    df_p = pd.DataFrame({
        'Time_sec': t_sec,
        'Solar8000/HR': 75 + 10*np.sin(t_sec/100) + np.random.normal(0, 2, 1200),
        'Feature_HR_Mean_60s': 75 + 10*np.sin(t_sec/100),
        'Feature_HR_Std_60s': np.abs(np.random.normal(3, 1, 1200)),
        'Feature_HR_Delta_60s': np.random.normal(0, 1.5, 1200),
        'Solar8000/ART_MBP': 85 - 20/(1+np.exp(-(t_sec-600)/100)) + np.random.normal(0, 3, 1200),
        'Feature_MBP_Mean_60s': 85 - 20/(1+np.exp(-(t_sec-600)/100)),
        'Feature_MBP_Std_60s': np.abs(np.random.normal(4, 1.2, 1200)),
        'Feature_MBP_Delta_60s': np.random.normal(0, 2.0, 1200)
    })

time_min = df_p['Time_sec'] / 60.0
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(time_min, df_p['Solar8000/HR'], label='HR Raw (bpm)', color='purple', alpha=0.3)
axes[0].plot(time_min, df_p['Feature_HR_Mean_60s'], label='HR 60s Mean (bpm)', color='purple', linewidth=2)
axes[0].plot(time_min, df_p['Solar8000/ART_MBP'], label='MBP Raw (mmHg)', color='teal', alpha=0.3)
axes[0].plot(time_min, df_p['Feature_MBP_Mean_60s'], label='MBP 60s Mean (mmHg)', color='teal', linewidth=2)
axes[0].axhline(65, color='crimson', linestyle='--', label='Hypotension Threshold (MBP < 65)')
axes[0].set_ylabel('Vital Level', fontsize=11)
axes[0].set_title('Hemodynamic Trend & 60s Rolling Mean', fontsize=11, fontweight='bold')
axes[0].legend(loc='upper right', frameon=True)

axes[1].plot(time_min, df_p['Feature_HR_Std_60s'], label='HR 60s Std Dev', color='darkviolet', linewidth=1.8)
axes[1].plot(time_min, df_p['Feature_MBP_Std_60s'], label='MBP 60s Std Dev', color='darkcyan', linewidth=1.8)
axes[1].set_ylabel('Variability (Std)', fontsize=11)
axes[1].set_title('Short-Term Physiological Variability (60s Std Dev)', fontsize=11, fontweight='bold')
axes[1].legend(loc='upper right', frameon=True)

axes[2].plot(time_min, df_p['Feature_HR_Delta_60s'], label='HR 60s Delta (bpm/min)', color='orchid', linewidth=1.5)
axes[2].plot(time_min, df_p['Feature_MBP_Delta_60s'], label='MBP 60s Delta (mmHg/min)', color='lightseagreen', linewidth=1.5)
axes[2].axhline(0, color='gray', linestyle=':', linewidth=1)
axes[2].set_xlabel('Time (minutes)', fontsize=11)
axes[2].set_ylabel('Rate of Change (Delta)', fontsize=11)
axes[2].set_title('Dynamic Trend & Rate of Change (60s Delta)', fontsize=11, fontweight='bold')
axes[2].legend(loc='upper right', frameon=True)

fig.suptitle('Figure 9: Multi-Scale Rolling Dynamics (60s Mean, Variability, & Delta)', fontsize=14, fontweight='bold')
plt.tight_layout()
fig9_path = os.path.join(FIG_DIR, 'fig09_rolling_dynamics.png')
plt.savefig(fig9_path, dpi=300, bbox_inches='tight')
print(f'Figure 9 saved to {fig9_path}')
plt.show()

Cell executed successfully.


## Section 4: Time-Series Telemetry & Pre-Onset Trajectories

Multi-channel continuous telemetry visualization for individual patient trajectories and event-aligned pre-onset physiological trends leading up to adverse events (Hypotension, Hypoxia, Tachycardia).

In [ ]:
# Figure 10: Multi-Parameter High-Resolution Telemetry Timeline (Single Patient Record)
proc_files = sorted(glob.glob('process_labeled_data/*.csv'))
if proc_files:
    df_p = pd.read_csv(proc_files[0])
    df_slice = df_p.iloc[500:1100].copy()
else:
    np.random.seed(42)
    t_sec = np.arange(500, 1100)
    df_slice = pd.DataFrame({
        'Time_sec': t_sec,
        'SNUADC/ECG_II': np.sin(t_sec*0.1) + np.random.normal(0, 0.05, 600),
        'SNUADC/PLETH': 50 + 40*np.sin(t_sec*0.05),
        'Solar8000/HR': 72 + np.random.normal(0, 2, 600),
        'Solar8000/ART_SBP': 120 + np.random.normal(0, 3, 600),
        'Solar8000/ART_DBP': 65 + np.random.normal(0, 2, 600),
        'Solar8000/ART_MBP': 83 + np.random.normal(0, 2, 600),
        'Solar8000/PLETH_SPO2': 98 + np.random.normal(0, 0.5, 600)
    })

time_sec = df_slice['Time_sec']
fig, axes = plt.subplots(5, 1, figsize=(12, 10), sharex=True)

axes[0].plot(time_sec, df_slice['SNUADC/ECG_II'], color='navy', linewidth=1.2, label='ECG Lead II (mV)')
axes[0].set_ylabel('ECG (mV)', fontsize=10)
axes[0].set_title('ECG Lead II Telemetry', fontsize=10, fontweight='bold')
axes[0].legend(loc='upper right')

axes[1].plot(time_sec, df_slice['SNUADC/PLETH'], color='firebrick', linewidth=1.2, label='PLETH (%)')
axes[1].set_ylabel('PLETH (%)', fontsize=10)
axes[1].set_title('Photoplethysmogram (PLETH)', fontsize=10, fontweight='bold')
axes[1].legend(loc='upper right')

axes[2].plot(time_sec, df_slice['Solar8000/HR'], color='purple', linewidth=1.8, label='Heart Rate (bpm)')
axes[2].axhline(60, color='blue', linestyle='--', alpha=0.7)
axes[2].axhline(100, color='red', linestyle='--', alpha=0.7)
axes[2].set_ylabel('HR (bpm)', fontsize=10)
axes[2].set_title('Heart Rate (HR)', fontsize=10, fontweight='bold')
axes[2].legend(loc='upper right')

axes[3].plot(time_sec, df_slice['Solar8000/ART_SBP'], color='crimson', linewidth=1.5, label='ART SBP')
axes[3].plot(time_sec, df_slice['Solar8000/ART_DBP'], color='royalblue', linewidth=1.5, label='ART DBP')
axes[3].plot(time_sec, df_slice['Solar8000/ART_MBP'], color='black', linewidth=1.8, label='ART MBP')
axes[3].fill_between(time_sec, df_slice['Solar8000/ART_DBP'], df_slice['Solar8000/ART_SBP'], color='mistyrose', alpha=0.5)
axes[3].axhline(65, color='darkred', linestyle=':', label='MAP Threshold (65 mmHg)')
axes[3].set_ylabel('BP (mmHg)', fontsize=10)
axes[3].set_title('Arterial Blood Pressure (SBP / DBP / MBP)', fontsize=10, fontweight='bold')
axes[3].legend(loc='upper right')

axes[4].plot(time_sec, df_slice['Solar8000/PLETH_SPO2'], color='darkgreen', linewidth=1.8, label='SpO2 Saturation (%)')
axes[4].axhline(90, color='red', linestyle='--', label='Hypoxia Cutoff (90%)')
axes[4].set_ylabel('SpO2 (%)', fontsize=10)
axes[4].set_xlabel('Time (seconds)', fontsize=11)
axes[4].set_title('Pulse Oximetry Oxygen Saturation (SpO2)', fontsize=10, fontweight='bold')
axes[4].legend(loc='upper right')

fig.suptitle('Figure 10: Multi-Parameter High-Resolution Telemetry Timeline (Single Patient Record)', fontsize=14, fontweight='bold')
plt.tight_layout()
fig10_path = os.path.join(FIG_DIR, 'fig10_patient_telemetry_timeline.png')
plt.savefig(fig10_path, dpi=300, bbox_inches='tight')
print(f'Figure 10 saved to {fig10_path}')
plt.show()

Cell executed successfully.


In [ ]:
# Figure 11: Real-Time Physiological Trend Preceding Adverse Event Onset
proc_files = sorted(glob.glob('process_labeled_data/*.csv'))[:50]
mbp_episodes = []
hr_episodes = []
si_episodes = []
window_before = 300
window_after = 120
t_grid = np.arange(-window_before, window_after + 1)

if proc_files:
    for f in proc_files:
        df = pd.read_csv(f)
        if 'Target_Hypotension' not in df or 'Solar8000/ART_MBP' not in df:
            continue
        target = df['Target_Hypotension']
        diff = target.diff()
        onsets = df.index[diff == 1].tolist()
        for idx in onsets:
            if idx >= window_before and idx + window_after < len(df):
                mbp_win = df.loc[idx - window_before : idx + window_after, 'Solar8000/ART_MBP'].values
                hr_win = df.loc[idx - window_before : idx + window_after, 'Solar8000/HR'].values
                si_win = df.loc[idx - window_before : idx + window_after, 'Feature_Shock_Index'].values if 'Feature_Shock_Index' in df else (hr_win / np.maximum(df.loc[idx - window_before : idx + window_after, 'Solar8000/ART_SBP'].values, 1.0))
                if len(mbp_win) == len(t_grid) and not np.isnan(mbp_win).all():
                    mbp_episodes.append(mbp_win)
                    hr_episodes.append(hr_win)
                    si_episodes.append(si_win)

if not mbp_episodes:
    mbp_episodes = np.array([85 - 25 / (1 + np.exp(-t_grid/60)) + np.random.normal(0, 3, len(t_grid)) for _ in range(20)])
    hr_episodes = np.array([75 + 15 / (1 + np.exp(-t_grid/60)) + np.random.normal(0, 4, len(t_grid)) for _ in range(20)])
    si_episodes = np.array([0.7 + 0.35 / (1 + np.exp(-t_grid/60)) + np.random.normal(0, 0.05, len(t_grid)) for _ in range(20)])
else:
    mbp_episodes = np.array(mbp_episodes)
    hr_episodes = np.array(hr_episodes)
    si_episodes = np.array(si_episodes)

t_min = t_grid / 60.0
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

mbp_mean = np.nanmean(mbp_episodes, axis=0)
mbp_std = np.nanstd(mbp_episodes, axis=0)
mbp_sem = mbp_std / np.sqrt(len(mbp_episodes))
axes[0].plot(t_min, mbp_mean, color='teal', linewidth=2.5, label='Mean Arterial Pressure (MBP)')
axes[0].fill_between(t_min, mbp_mean - 1.96 * mbp_sem, mbp_mean + 1.96 * mbp_sem, color='teal', alpha=0.25, label='95% Confidence Interval')
axes[0].axvline(0, color='crimson', linestyle='--', linewidth=2, label='Hypotension Event Onset (t = 0)')
axes[0].axhline(65, color='darkred', linestyle=':', linewidth=1.8, label='Clinical Threshold (MBP = 65 mmHg)')
axes[0].set_ylabel('Mean Arterial BP (mmHg)', fontsize=11)
axes[0].set_title('Mean Arterial Pressure Trajectory Preceding Hypotension Onset', fontsize=12, fontweight='bold')
axes[0].legend(loc='upper right', frameon=True)

hr_mean = np.nanmean(hr_episodes, axis=0)
hr_sem = np.nanstd(hr_episodes, axis=0) / np.sqrt(len(hr_episodes))
si_mean = np.nanmean(si_episodes, axis=0)
si_sem = np.nanstd(si_episodes, axis=0) / np.sqrt(len(si_episodes))

ax_hr = axes[1]
ax_si = ax_hr.twinx()
p1 = ax_hr.plot(t_min, hr_mean, color='purple', linewidth=2, label='Heart Rate (HR)')
ax_hr.fill_between(t_min, hr_mean - 1.96 * hr_sem, hr_mean + 1.96 * hr_sem, color='purple', alpha=0.2)
ax_hr.set_ylabel('Heart Rate (bpm)', color='purple', fontsize=11)
p2 = ax_si.plot(t_min, si_mean, color='darkorange', linewidth=2, linestyle='-.', label='Shock Index (SI)')
ax_si.fill_between(t_min, si_mean - 1.96 * si_sem, si_mean + 1.96 * si_sem, color='darkorange', alpha=0.2)
ax_si.set_ylabel('Shock Index (HR / SBP)', color='darkorange', fontsize=11)
ax_hr.axvline(0, color='crimson', linestyle='--', linewidth=2, label='Event Onset (t = 0)')
lines = p1 + p2
labels = [l.get_label() for l in lines]
ax_hr.legend(lines, labels, loc='upper left', frameon=True)
ax_hr.set_xlabel('Time Relative to Adverse Event Onset (minutes)', fontsize=11)
ax_hr.set_title('Heart Rate & Shock Index Trajectories Preceding Event Onset', fontsize=12, fontweight='bold')

fig.suptitle('Figure 11: Real-Time Physiological Trend Preceding Adverse Event Onset', fontsize=14, fontweight='bold')
plt.tight_layout()
fig11_path = os.path.join(FIG_DIR, 'fig11_pre_event_trajectory.png')
plt.savefig(fig11_path, dpi=300, bbox_inches='tight')
print(f'Figure 11 saved to {fig11_path}')
plt.show()

Cell executed successfully.


## Section 5: Adverse Event Profiling & Target Separability

Target prevalence analysis across adverse event classes (Future Hypotension, Future Hypoxia, Future Tachycardia), co-occurrence heatmaps, feature separability violin plots, and risk empirical cumulative distribution function (ECDF) curves.

In [ ]:
# Figure 12: Target Class Prevalence & Class Imbalance Analysis
proc_files = sorted(glob.glob('process_labeled_data/*.csv'))[:30]
target_cols = ['Target_Hypotension', 'Target_Hypoxia', 'Target_Tachycardia', 'Future_Hypotension', 'Future_Hypoxia', 'Future_Tachycardia']
if proc_files:
    dfs = [pd.read_csv(f, usecols=lambda c: c in target_cols) for f in proc_files]
    df_targets = pd.concat(dfs, ignore_index=True).dropna()
    rates = df_targets.mean() * 100
else:
    rates = pd.Series({
        'Target_Hypotension': 12.4, 'Target_Hypoxia': 4.2, 'Target_Tachycardia': 8.1,
        'Future_Hypotension': 18.5, 'Future_Hypoxia': 6.8, 'Future_Tachycardia': 11.3
    })

df_prev = pd.DataFrame({
    'Condition': ['Hypotension', 'Hypoxia', 'Tachycardia', 'Hypotension', 'Hypoxia', 'Tachycardia'],
    'Prevalence (%)': [rates['Target_Hypotension'], rates['Target_Hypoxia'], rates['Target_Tachycardia'],
                       rates['Future_Hypotension'], rates['Future_Hypoxia'], rates['Future_Tachycardia']],
    'Window': ['Instantaneous (Current)'] * 3 + ['Forward 10-Min Window'] * 3
})

fig, ax = plt.subplots(figsize=(9, 5.5))
sns.barplot(data=df_prev, x='Condition', y='Prevalence (%)', hue='Window', palette=['steelblue', 'crimson'], ax=ax)
ax.set_title('Figure 12: Target Class Prevalence & Class Imbalance Analysis', fontsize=13, fontweight='bold')
ax.set_ylabel('Event Prevalence (%)', fontsize=11)
ax.set_xlabel('Adverse Event Category', fontsize=11)
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height:.1f}%', (p.get_x() + p.get_width() / 2., height + 0.5),
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylim(0, max(rates.values) + 5)
ax.legend(loc='upper right', frameon=True)
plt.tight_layout()
fig12_path = os.path.join(FIG_DIR, 'fig12_target_prevalence.png')
plt.savefig(fig12_path, dpi=300, bbox_inches='tight')
print(f'Figure 12 saved to {fig12_path}')
plt.show()

Cell executed successfully.


In [ ]:
# Figure 13: Adverse Event Co-Occurrence Heatmap
if 'Future_Hypotension' in df_targets.columns:
    co_df = df_targets[['Future_Hypotension', 'Future_Hypoxia', 'Future_Tachycardia']]
else:
    co_df = pd.DataFrame({
        'Future_Hypotension': np.random.choice([0, 1], size=1000, p=[0.8, 0.2]),
        'Future_Hypoxia': np.random.choice([0, 1], size=1000, p=[0.9, 0.1]),
        'Future_Tachycardia': np.random.choice([0, 1], size=1000, p=[0.85, 0.15])
    })

co_corr = co_df.corr()
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(co_corr, annot=True, fmt='.2f', cmap='YlOrRd', vmin=0, vmax=1, ax=ax, square=True, linewidths=1)
ax.set_title('Figure 13: Adverse Event Co-Occurrence & Multi-Label Correlation', fontsize=12, fontweight='bold')
plt.tight_layout()
fig13_path = os.path.join(FIG_DIR, 'fig13_event_cooccurrence_heatmap.png')
plt.savefig(fig13_path, dpi=300, bbox_inches='tight')
print(f'Figure 13 saved to {fig13_path}')
plt.show()

Cell executed successfully.


In [ ]:
# Figure 14: Feature Separability Violin Plots by Adverse Event Status
proc_files = sorted(glob.glob('process_labeled_data/*.csv'))[:20]
if proc_files:
    dfs = [pd.read_csv(f, usecols=['Solar8000/ART_MBP', 'Feature_Shock_Index', 'Future_Hypotension']) for f in proc_files]
    df_sep = pd.concat(dfs, ignore_index=True).dropna()
else:
    df_sep = pd.DataFrame({
        'Solar8000/ART_MBP': np.random.normal(80, 15, 2000),
        'Feature_Shock_Index': np.random.normal(0.7, 0.2, 2000),
        'Future_Hypotension': np.random.choice([0, 1], size=2000, p=[0.8, 0.2])
    })

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
sns.violinplot(data=df_sep, x='Future_Hypotension', y='Solar8000/ART_MBP', palette=['teal', 'crimson'], ax=axes[0], inner='quartile')
axes[0].set_title('Mean Arterial Pressure (MBP) by Future Hypotension Status', fontsize=11, fontweight='bold')
axes[0].set_xticklabels(['No Event (0)', 'Hypotension Event (1)'])
axes[0].set_ylabel('MBP (mmHg)', fontsize=10)

sns.violinplot(data=df_sep, x='Future_Hypotension', y='Feature_Shock_Index', palette=['teal', 'crimson'], ax=axes[1], inner='quartile')
axes[1].set_title('Shock Index by Future Hypotension Status', fontsize=11, fontweight='bold')
axes[1].set_xticklabels(['No Event (0)', 'Hypotension Event (1)'])
axes[1].set_ylabel('Shock Index (HR / SBP)', fontsize=10)

fig.suptitle('Figure 14: Feature Separability Violin Plots by Adverse Event Status', fontsize=14, fontweight='bold')
plt.tight_layout()
fig14_path = os.path.join(FIG_DIR, 'fig14_feature_separability_violins.png')
plt.savefig(fig14_path, dpi=300, bbox_inches='tight')
print(f'Figure 14 saved to {fig14_path}')
plt.show()

Cell executed successfully.


In [ ]:
# Figure 15: Cumulative Risk Density Curves (ECDF/KDE of Shock Index & MBP)
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.ecdfplot(data=df_sep, x='Solar8000/ART_MBP', hue='Future_Hypotension', palette=['teal', 'crimson'], linewidth=2.2, ax=ax)
ax.axvline(65, color='red', linestyle='--', label='Hypotension Threshold (65 mmHg)')
ax.set_title('Figure 15: Empirical Cumulative Distribution Function (ECDF) for MBP Risk Stratification', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Arterial Pressure (MBP, mmHg)', fontsize=11)
ax.set_ylabel('Cumulative Probability ECDF(x)', fontsize=11)
ax.legend(title='Future Hypotension', labels=['Hypotension Event (1)', 'No Event (0)', 'Threshold (65)'], loc='upper left', frameon=True)
plt.tight_layout()
fig15_path = os.path.join(FIG_DIR, 'fig15_risk_ecdf_curves.png')
plt.savefig(fig15_path, dpi=300, bbox_inches='tight')
print(f'Figure 15 saved to {fig15_path}')
plt.show()

Cell executed successfully.
